# Dataset construction and response generation

Use the existing Python classes to export TACO prompts, load a small sample,
and optionally generate responses with experimental statistics.

The notebook contains configuration and orchestration. Dataset operations belong
to `TacoDataset` and `PromptOperation`; inference statistics belong to
`ResponseGenerator`; persistence and recovery belong to `BatchGenerator`.
No generated solution is executed here.


## 1. Repository setup

In [1]:
# Standard library: resolve the repository before importing project modules.
import json
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
for directory in (working_directory, *working_directory.parents):
    if (directory / "modeling/src/data/export_prompts.py").is_file():
        PROJECT_ROOT = directory
        break
else:
    raise FileNotFoundError("Open this notebook inside prompt-difficulty-predictor")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODELING_DIR = PROJECT_ROOT / "modeling"
DATA_DIR = MODELING_DIR / "data"


## 2. Imports

Import only the classes available in the current Python modules.

In [2]:
from itertools import islice

from modeling.src.data.export_prompts import PromptOperation
from modeling.src.data.taco_dataset import TacoDataset
from modeling.src.evaluation.batch import BatchGenerator
from modeling.src.evaluation.generate_responses import GenerationConfig, ResponseGenerator


/Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/src/evaluation/hardware_monitor.py:9: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


## 3. Configuration

Reuse the recorded pilot to inspect saved responses. Choose a new `RUN_ID` if
any inference setting or the serialized configuration has changed.
`RUN_OLLAMA=False` keeps generation disabled when running all cells.


In [3]:
PROMPTS_PATH = DATA_DIR / "processed/prompts.jsonl"
REBUILD_PROMPTS = False
MAX_PROMPTS = 3
RUN_OLLAMA = False

MODEL = "qwen3:8b"  # Use llama3.1:8b if that is the installed target model.
OPTIONS = {"temperature": 0.0, "top_p": 1.0, "seed": 42, "num_ctx": 4096, "num_predict": 512}
SYSTEM_PROMPT = (
    "Solve the programming problem in Python 3. Return exactly one fenced "
    "python code block containing the complete solution, without explanations. "
    "Follow the input/output format and function signature in the statement. "
    "Do not access files, the network or external services."
)

RUN_ID = "pilot_20261026_qwen3_notebook"
RUN_DIR = MODELING_DIR / "experiments/runs" / RUN_ID
RESPONSES_PATH = RUN_DIR / "responses.jsonl"

config = GenerationConfig(model=MODEL, options=OPTIONS, system=SYSTEM_PROMPT)
dataset = TacoDataset(DATA_DIR / "raw/TACO")
prompt_operation = PromptOperation(data_dir=DATA_DIR)


## 4. Export and load prompts

`PromptOperation.export_prompts()` validates and deduplicates the questions
provided by `TacoDataset.questions()` and writes the existing prompt-only JSONL
format. Reuse that file unless rebuilding is requested.

Load only `MAX_PROMPTS` strings into memory. `BatchGenerator.run()` already
accepts strings and computes their identifiers, so the notebook does not need
to duplicate hashing, validation or history handling.


In [4]:
if REBUILD_PROMPTS or not PROMPTS_PATH.exists():
    summary = prompt_operation.export_prompts(dataset.questions(), PROMPTS_PATH)
    print(summary)
else:
    print(f"Reusing: {PROMPTS_PATH}")

with PROMPTS_PATH.open(encoding="utf-8") as stream:
    selected_prompts = [
        json.loads(line)["prompt"]
        for line in islice((line for line in stream if line.strip()), MAX_PROMPTS)
    ]

print(f"Loaded prompts: {len(selected_prompts)}")
for index, prompt in enumerate(selected_prompts, start=1):
    print(index, prompt[:120].replace("\n", " "))


Reusing: /Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/data/processed/prompts.jsonl
Loaded prompts: 3
1 This is an interactive problem.  In good old times dwarves tried to develop extrasensory abilities:    * Exactly n dwarv
2 There are $n$ candy boxes in front of Tania. The boxes are arranged in a row from left to right, numbered from $1$ to $n
3 Little Petya likes to play a lot. Most of all he likes to play a game «Holes». This is a game for one person with follow


## 5. Generate responses and save statistics

`ResponseGenerator` collects the response, provider timings, token counts,
throughput and any configured hardware samples. `BatchGenerator` persists each
attempt in `responses.jsonl` and manages retries, interrupted writes and resume.

The notebook does not implement its own provider calls, statistics calculations
or response writer. Hardware monitoring is optional and is not configured here.


In [5]:
if RUN_OLLAMA:
    generator = ResponseGenerator(config)
    batch = BatchGenerator(generator, RUN_DIR)
    print(batch.run(selected_prompts))
else:
    print("Inference disabled. Set RUN_OLLAMA=True to generate the selected prompts.")


Inference disabled. Set RUN_OLLAMA=True to generate the selected prompts.


## 6. Inspect saved responses

Read a small preview without loading the complete response history.

In [6]:
if RESPONSES_PATH.exists():
    with RESPONSES_PATH.open(encoding="utf-8") as stream:
        for line in islice((line for line in stream if line.strip()), 3):
            record = json.loads(line)
            print(record["prompt_id"][:12], record["status"], record.get("done_reason"))
            print(record.get("response", record.get("error", ""))[:200])
else:
    print("No responses saved for this run yet.")


e0e1a40c9f75 generated stop
```python
import sys
import random

def main():
    import sys
    input = sys.stdin.readline

    n = int(input())
    points = []
    for _ in range(n):
        x = random.randint(0, 10**9)
        
f6e6e2933845 generated stop
```python
import sys
import heapq

def main():
    import sys
    input = sys.stdin.read
    data = input().split()
    
    n = int(data[0])
    s = int(data[1])
    k = int(data[2])
    
    r = lis
b024a891ac3b generated stop
```python
import sys
input = sys.stdin.read

def main():
    data = input().split()
    idx = 0
    N = int(data[idx])
    idx += 1
    M = int(data[idx])
    idx += 1
    a = list(map(int, data[idx:i


## Existing run artifacts

```text
modeling/experiments/runs/<RUN_ID>/
├── config.json
└── responses.jsonl
```

This notebook does not prepare or execute sandbox tasks: no sandbox module is
present in the current Python source tree.
